In [1]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, avg, count


In [2]:
spark = SparkSession.builder.appName("Basics").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/11 21:26:56 WARN Utils: Your hostname, Manshus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.221.38.203 instead (on interface en0)
25/10/11 21:26:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/11 21:26:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### To Read Data from Datasets

In [3]:
df = spark.read.csv("../datasets/testing.csv", header=True, inferSchema=True)

In [4]:
df.show()

+----------------+------+
|            Name|Number|
+----------------+------+
|   Manshu Sharma|     1|
|Swayista T Ahmed|     2|
|      Svastikkka|  NULL|
+----------------+------+



### Creating DataFrames

In [5]:
data = [("Alice", 25, "F"), ("Bob", 30, "M"), ("Cathy", 27, "F")]
df = spark.createDataFrame(data, ["Name", "Age", "Gender"])
df.show() # Display top 20 rows

+-----+---+------+
| Name|Age|Gender|
+-----+---+------+
|Alice| 25|     F|
|  Bob| 30|     M|
|Cathy| 27|     F|
+-----+---+------+



In [6]:
df.printSchema()    # Show schema

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)



In [7]:
df.head(1)          # First row as a list


[Row(Name='Alice', Age=25, Gender='F')]

In [8]:
df.first()          # First row as Row object

Row(Name='Alice', Age=25, Gender='F')

In [9]:
df.count()          # Number of rows

3

In [10]:
df.describe().show() # Summary statistics


+-------+-----+------------------+------+
|summary| Name|               Age|Gender|
+-------+-----+------------------+------+
|  count|    3|                 3|     3|
|   mean| NULL|27.333333333333332|  NULL|
| stddev| NULL| 2.516611478423583|  NULL|
|    min|Alice|                25|     F|
|    max|Cathy|                30|     M|
+-------+-----+------------------+------+



### Selecting Columns

In [11]:
df.select("Name").show()
df.select(col("Name"), col("Age") + 1).show()

+-----+
| Name|
+-----+
|Alice|
|  Bob|
|Cathy|
+-----+

+-----+---------+
| Name|(Age + 1)|
+-----+---------+
|Alice|       26|
|  Bob|       31|
|Cathy|       28|
+-----+---------+



### Filtering / Where

In [12]:
df.filter(df.Age > 25).show()
df.where((col("Age") > 25) & (col("Gender") == "F")).show()

+-----+---+------+
| Name|Age|Gender|
+-----+---+------+
|  Bob| 30|     M|
|Cathy| 27|     F|
+-----+---+------+

+-----+---+------+
| Name|Age|Gender|
+-----+---+------+
|Cathy| 27|     F|
+-----+---+------+



### Adding / Dropping Columns

In [13]:
# Add a new column
df = df.withColumn("AgePlus10", col("Age") + 10)
df.show()

+-----+---+------+---------+
| Name|Age|Gender|AgePlus10|
+-----+---+------+---------+
|Alice| 25|     F|       35|
|  Bob| 30|     M|       40|
|Cathy| 27|     F|       37|
+-----+---+------+---------+



In [14]:
# Drop a column
df = df.drop("AgePlus10")
df.show()

+-----+---+------+
| Name|Age|Gender|
+-----+---+------+
|Alice| 25|     F|
|  Bob| 30|     M|
|Cathy| 27|     F|
+-----+---+------+



### Renaming Columns

In [15]:
df = df.withColumnRenamed("Gender", "Sex")
df.show()

+-----+---+---+
| Name|Age|Sex|
+-----+---+---+
|Alice| 25|  F|
|  Bob| 30|  M|
|Cathy| 27|  F|
+-----+---+---+



### Sorting / Ordering

In [16]:
df.orderBy("Age").show()          # Ascending
df.orderBy(col("Age").desc()).show()  # Descending

+-----+---+---+
| Name|Age|Sex|
+-----+---+---+
|Alice| 25|  F|
|Cathy| 27|  F|
|  Bob| 30|  M|
+-----+---+---+

+-----+---+---+
| Name|Age|Sex|
+-----+---+---+
|  Bob| 30|  M|
|Cathy| 27|  F|
|Alice| 25|  F|
+-----+---+---+



### Distinct & Drop Duplicates

In [17]:
df_distinct = df.distinct()
df_distinct.show()

+-----+---+---+
| Name|Age|Sex|
+-----+---+---+
|Alice| 25|  F|
|  Bob| 30|  M|
|Cathy| 27|  F|
+-----+---+---+



In [18]:
df_no_dupes = df.dropDuplicates(["Name"])
df_no_dupes.show()

+-----+---+---+
| Name|Age|Sex|
+-----+---+---+
|Alice| 25|  F|
|  Bob| 30|  M|
|Cathy| 27|  F|
+-----+---+---+



### Group By & Aggregations

In [19]:
df.groupBy("Sex").count().show()

+---+-----+
|Sex|count|
+---+-----+
|  F|    2|
|  M|    1|
+---+-----+



In [20]:
df.groupBy("Sex").agg(avg("Age").alias("avg_age")).show()

+---+-------+
|Sex|avg_age|
+---+-------+
|  F|   26.0|
|  M|   30.0|
+---+-------+



### Joining DataFrames

In [21]:
data2 = [("Alice", "NY"), ("Bob", "LA"), ("David", "SF")]
df2 = spark.createDataFrame(data2, ["Name", "City"])

In [22]:
# Inner Join
df.join(df2, on="Name", how="inner").show()

+-----+---+---+----+
| Name|Age|Sex|City|
+-----+---+---+----+
|Alice| 25|  F|  NY|
|  Bob| 30|  M|  LA|
+-----+---+---+----+



In [23]:
# Left/Right/Outer Join
df.join(df2, on="Name", how="left").show()

+-----+---+---+----+
| Name|Age|Sex|City|
+-----+---+---+----+
|Alice| 25|  F|  NY|
|  Bob| 30|  M|  LA|
|Cathy| 27|  F|NULL|
+-----+---+---+----+



### Union & Intersect

In [24]:
df1 = spark.createDataFrame([("Tom", 40)], ["Name", "Age"])
df2 = spark.createDataFrame([("Jerry", 35)], ["Name", "Age"])

df_union = df1.union(df2)
df_union.show()

df_intersect = df_union.intersect(df1)
df_intersect.show()
df_intersect = df_union.intersect(df2)
df_intersect.show()

+-----+---+
| Name|Age|
+-----+---+
|  Tom| 40|
|Jerry| 35|
+-----+---+

+----+---+
|Name|Age|
+----+---+
| Tom| 40|
+----+---+

+-----+---+
| Name|Age|
+-----+---+
|Jerry| 35|
+-----+---+



### Handling Missing Data

In [25]:
df_na = spark.createDataFrame([("Alice", None), ("Bob", 30)], ["Name", "Age"])

df_na.dropna().show()                # Drop rows with null
df_na.fillna({"Age": 0}).show()      # Fill nulls

+----+---+
|Name|Age|
+----+---+
| Bob| 30|
+----+---+

+-----+---+
| Name|Age|
+-----+---+
|Alice|  0|
|  Bob| 30|
+-----+---+



In [26]:
df_na = spark.createDataFrame([("Alice", 0), ("Bob", 30)], ["Name", "Age"])
df_na.replace(0, 100, subset=["Age"]).show()

+-----+---+
| Name|Age|
+-----+---+
|Alice|100|
|  Bob| 30|
+-----+---+



### Conditional Column (with when)

In [27]:
df.withColumn("Adult", when(col("Age") >= 30, "Yes").otherwise("No")).show()

+-----+---+---+-----+
| Name|Age|Sex|Adult|
+-----+---+---+-----+
|Alice| 25|  F|   No|
|  Bob| 30|  M|  Yes|
|Cathy| 27|  F|   No|
+-----+---+---+-----+



### String Operations

In [28]:
from pyspark.sql.functions import upper, lower, concat_ws

df.select(upper(col("Name")), lower(col("Name"))).show()
df.select(concat_ws("-", col("Name"), col("Sex"))).show()


+-----------+-----------+
|upper(Name)|lower(Name)|
+-----------+-----------+
|      ALICE|      alice|
|        BOB|        bob|
|      CATHY|      cathy|
+-----------+-----------+

+-----------------------+
|concat_ws(-, Name, Sex)|
+-----------------------+
|                Alice-F|
|                  Bob-M|
|                Cathy-F|
+-----------------------+



### Type Casting

In [29]:
# Type Casting means converting the data type of a column from one type to another.
df.withColumn("AgeStr", col("Age").cast("string")).printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Sex: string (nullable = true)
 |-- AgeStr: string (nullable = true)



### Pivot / Melt (Reshape)

In [30]:
data = [("Alice", "Math", 85), ("Alice", "English", 78), ("Bob", "Math", 90)]
df2 = spark.createDataFrame(data, ["Name", "Subject", "Score"])

df_pivot = df2.groupBy("Name").pivot("Subject").avg("Score")
df_pivot.show()


+-----+-------+----+
| Name|English|Math|
+-----+-------+----+
|  Bob|   NULL|90.0|
|Alice|   78.0|85.0|
+-----+-------+----+



### Working with Dates

In [31]:
from pyspark.sql.functions import current_date, datediff

df_date = df.withColumn("Today", current_date())
df_date.show()
df_date.withColumn("DaysSinceToday", datediff(current_date(), col("Today"))).show()
df_date.show()

+-----+---+---+----------+
| Name|Age|Sex|     Today|
+-----+---+---+----------+
|Alice| 25|  F|2025-10-11|
|  Bob| 30|  M|2025-10-11|
|Cathy| 27|  F|2025-10-11|
+-----+---+---+----------+

+-----+---+---+----------+--------------+
| Name|Age|Sex|     Today|DaysSinceToday|
+-----+---+---+----------+--------------+
|Alice| 25|  F|2025-10-11|             0|
|  Bob| 30|  M|2025-10-11|             0|
|Cathy| 27|  F|2025-10-11|             0|
+-----+---+---+----------+--------------+

+-----+---+---+----------+
| Name|Age|Sex|     Today|
+-----+---+---+----------+
|Alice| 25|  F|2025-10-11|
|  Bob| 30|  M|2025-10-11|
|Cathy| 27|  F|2025-10-11|
+-----+---+---+----------+



### Collect / Convert to Pandas

In [32]:
# Collect rows to driver
rows = df.collect()
print(rows)

# Convert to Pandas
pdf = df.toPandas()
print(pdf.head())

[Row(Name='Alice', Age=25, Sex='F'), Row(Name='Bob', Age=30, Sex='M'), Row(Name='Cathy', Age=27, Sex='F')]
    Name  Age Sex
0  Alice   25   F
1    Bob   30   M
2  Cathy   27   F


### Caching & Persisting

In [33]:
df.cache()        # Cache in memory

DataFrame[Name: string, Age: bigint, Sex: string]

In [34]:
df.count()        # Action triggers cache

3

In [35]:
df.unpersist()    # Remove from cache

DataFrame[Name: string, Age: bigint, Sex: string]

### SQL Queries on DataFrames

In [36]:
df.createOrReplaceTempView("people")
spark.sql("SELECT Name, Age FROM people WHERE Age > 25").show()

+-----+---+
| Name|Age|
+-----+---+
|  Bob| 30|
|Cathy| 27|
+-----+---+

